In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
!pip install -q pydantic==2.12.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 11.6 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.


In [3]:
!pip install -qi https://test.pypi.org/simple/ rwkvx==0.1.0.dev5

In [4]:
!pip install -q rwkv huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.9/410.9 kB 6.7 MB/s eta 0:00:00a 0:00:01


In [5]:
import os
import gc

import huggingface_hub
import torch

In [6]:
from rwkvx.conversation import (
    ConversationManager,
    FormatPromptTemplate,
    HistoryManager,
    Message,
    OutputParser,
    PromptRenderer,
)

In [7]:
os.environ["RWKV_V7_ON"] = '1'
os.environ["RWKV_JIT_ON"] = '1'
os.environ["RWKV_CUDA_ON"] = '1' # if '1' then use CUDA kernel for seq mode (much faster)

from rwkv.model import RWKV
from rwkv.utils import PIPELINE, PIPELINE_ARGS

Using /root/.cache/torch_extensions/py311_cu124 as PyTorch extensions root...
Creating extension directory /root/.cache/torch_extensions/py311_cu124/wkv_cuda...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py311_cu124/wkv_cuda/build.ninja...
/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module wkv_cuda...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


[1/4] c++ -MMD -MF gemm_fp16_cublas.o.d -DTORCH_EXTENSION_NAME=wkv_cuda -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/TH -isystem /usr/local/lib/python3.11/dist-packages/torch/include/THC -isystem /usr/local/cuda/include -isystem /usr/include/python3.11 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -c /usr/local/lib/python3.11/dist-packages/rwkv/cuda/gemm_fp16_cublas.cpp -o gemm_fp16_cublas.o 
[2/4] c++ -MMD -MF wrapper.o.d -DTORCH_EXTENSION_NAME=wkv_cuda -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include -isystem /usr/local/lib/python3.11/dist-p

Loading extension module wkv_cuda...
Using /root/.cache/torch_extensions/py311_cu124 as PyTorch extensions root...
Creating extension directory /root/.cache/torch_extensions/py311_cu124/wkv7s...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py311_cu124/wkv7s/build.ninja...
/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module wkv7s...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


[1/3] c++ -MMD -MF rwkv7_op.o.d -DTORCH_EXTENSION_NAME=wkv7s -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/TH -isystem /usr/local/lib/python3.11/dist-packages/torch/include/THC -isystem /usr/local/cuda/include -isystem /usr/include/python3.11 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -c /usr/local/lib/python3.11/dist-packages/rwkv/cuda/rwkv7_op.cpp -o rwkv7_op.o 
[2/3] /usr/local/cuda/bin/nvcc --generate-dependencies-with-compile --dependency-output rwkv7.cuda.o.d -DTORCH_EXTENSION_NAME=wkv7s -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include

Loading extension module wkv7s...


In [9]:
# model_title = "rwkv7-g1a4-2.9b-20251118-ctx8192"
model_title = "rwkv7-g0a4-7.2b-20251208-ctx8192"
# model_title = "rwkv7-g0b-13.3b-20251130-ctx8192"
model_path = huggingface_hub.hf_hub_download(repo_id="BlinkDL/rwkv7-g1", filename=f"{model_title}.pth")
rwkv_model = RWKV(model=model_path.replace('.pth',''), strategy='cuda fp16')
rwkv_pipeline = PIPELINE(rwkv_model, "rwkv_vocab_v20230424")

rwkv7-g0a4-7.2b-20251208-ctx8192.pth:   0%|          | 0.00/14.4G [00:00<?, ?B/s]

Loading /root/.cache/huggingface/hub/models--BlinkDL--rwkv7-g1/snapshots/b25bc6bccda433ca7c63eb571d6b12257784b5cc/rwkv7-g0a4-7.2b-20251208-ctx8192 (cuda fp16)



In [10]:
class RWKVModel:
    """
    A subclass of Model for RWKV7 inference, adapted from the experimental notebook.
    """
    def __init__(
        self,
        model,
        pipeline
    ):    
        # Setup RWKV
        self.CTX_LIMIT = 6000
        self.PENALTY_DECAY = 0.996
        self._model = model
        self._pipeline = pipeline

        # Model state
        self.state = None

    def reset_state(self) -> None:
        self.stare = None
    
    def generate(
        self,
        prompt: str,
        *,
        reset_state: bool = False,
    ) -> str:
        """Process the input messages and return the model's response."""
        if reset_state:
            self.reset_state()
        
        print(f"Input prompt:_", prompt)

        output = self.run_rwkv(prompt)
        output_text = output.strip()

        print(f"Output:_", output_text)
        
        return output_text

    def run_rwkv(
        self,
        ctx,
        token_count=2000,
        temperature=1.0,
        top_p=0.3,
        presencePenalty=0.5,
        countPenalty=0.5,
    ):    
        ctx = ctx.strip()
        all_tokens = []
        out_last = 0
        out_str = ''
        occurrence = {}
    
        for i in range(int(token_count)):
            input_ids = self._pipeline.encode(ctx)
            input_ids = input_ids[-self.CTX_LIMIT:] if i == 0 else [token]
            
            out, self.state = self._model.forward(input_ids, self.state)
    
            for n in occurrence:
                out[n] -= (presencePenalty + occurrence[n] * countPenalty)
    
            token = self._pipeline.sample_logits(
                out,
                temperature=temperature,
                top_p=top_p)
            if token in [261]:
                break
    
            all_tokens.append(token)
    
            for xxx in occurrence:
                occurrence[xxx] *= self.PENALTY_DECAY
    
            ttt = self._pipeline.decode([token])
            www = 1
            if ttt in ' \t0123456789':
                www = 0
            if token not in occurrence:
                occurrence[token] = www
            else:
                occurrence[token] += www
    
            tmp = self._pipeline.decode(all_tokens[out_last:])
            if '\ufffd' not in tmp:
                out_str += tmp
                out_last = i + 1

                if out_str.strip().endswith("\n\n"):
                    break
    
        # Cleanup and timing
        del out
        gc.collect()
        torch.cuda.empty_cache()
        
        return out_str.strip()

In [28]:
model = RWKVModel(model=rwkv_model, pipeline=rwkv_pipeline)
model.reset_state()

In [29]:
templates = {
        'system': FormatPromptTemplate('System: {text}\n\n'),
        'user': FormatPromptTemplate('User: {text}{think_mode}\n\n'),
        'assistant': FormatPromptTemplate('Assistant: {think_start}{think_text}{think_end}{text}\n\n'),
}

history = HistoryManager()

mgr = ConversationManager(
    templates=templates,
    history=history,
)

In [30]:
history.clear()
# mgr.add_user('Tell me a short joke in English.', metadata={'think_mode': ' think a bit'})
mgr.add_user('Tell me a short joke in English. Use only English in our conversation.')

In [31]:
mgr.build_rwkv7_prompt()

'User: Tell me a short joke in English. Use only English in our conversation.\n\nAssistant: <think>\n</think'

In [32]:
assistant_output = model.generate(mgr.build_rwkv7_prompt())
mgr.add_assistant_raw(assistant_output)

Input prompt:_ User: Tell me a short joke in English. Use only English in our conversation.

Assistant: <think>
</think
Output:_ >
Here's a short joke:
Why did the scarecrow win an award?  
Because he was outstanding in his field! 🌾


In [33]:
print(mgr.build_prompt())

User: Tell me a short joke in English. Use only English in our conversation.

Assistant: <think>
</think>
Here's a short joke:
Why did the scarecrow win an award?  
Because he was outstanding in his field! 🌾




In [34]:
mgr.add_user('Tell me another joke, similar to one before.')

In [35]:
print(mgr.build_rwkv7_prompt())
print('===')
mgr.build_rwkv7_prompt(user_assistant_count=1)

User: Tell me a short joke in English. Use only English in our conversation.

Assistant: <think>
</think>
Here's a short joke:
Why did the scarecrow win an award?  
Because he was outstanding in his field! 🌾

User: Tell me another joke, similar to one before.

Assistant: <think>
</think
===


'User: Tell me another joke, similar to one before.\n\nAssistant: <think>\n</think'

In [36]:
assistant_output = model.generate(mgr.build_rwkv7_prompt(user_assistant_count=1))
mgr.add_assistant_raw(assistant_output)

Input prompt:_ User: Tell me another joke, similar to one before.

Assistant: <think>
</think
Output:_ >
好的，我们来讲一个类似风格的笑话。这个笑话同样基于“数字”和“语言”的巧妙结合，希望您会喜欢。
---
### **笑话：关于“数”与“说”**
有一天，一位数学家走进了一家酒吧。他点了一杯啤酒，然后对酒保说：
**“你知道吗？我刚刚在心里算了算，如果我把这杯啤酒喝完，再加上你给我的小费，那么‘我’（I）就变成了‘二’（2）。”**
酒保愣了一下，然后若有所思地点点头。
数学家继续说：“而且更有趣的是，如果你把这句话翻译成中文——‘你知道吗？我刚刚在心里算了算，如果我把这杯啤酒喝完，再加上你给我的小费，那么‘我’（Wǒ）就变成了‘二’（Èr）。”——那么整个句子就变成了一个非常重要的数学定理：**1 + 1 = 2**。
酒保听完后哈哈大笑：“哇！所以你是说，这句话本身就是一个证明？”
数学家微笑着回答：“不仅仅是证明。它还告诉我们，无论用哪种语言表达，逻辑都是通用的。但最重要的是……” 他压低声音说道：“**现在你欠我两杯啤酒了！**”
---
### **为什么这个笑话像之前那些？**
1.  **核心元素相似：** 它依然围绕着“数字”（1, 2）和“语言”（英语/中文）展开。
2.  **双关语：** 利用了“数字”和“人名/代词”发音相似的特点进行幽默。
    *   英语中：`I (1) + 1 = 2 (Two)`
    *   中文中：`我 (Wǒ) + 1 = 二 (Èr)`
3.  **转折与反转：** 从简单的计算转向更深层次的哲学或逻辑思考（1+1=2），并最终将其引申到现实生活中的“债务”，增加了趣味性和意外感。
4.  **文化桥梁：** 它巧妙地连接了东西方文化（英语和中文），体现了数学作为全球通用语言的魅力。
希望这个笑话能让您会心一笑！


In [37]:
print(mgr.build_prompt())

User: Tell me a short joke in English. Use only English in our conversation.

Assistant: <think>
</think>
Here's a short joke:
Why did the scarecrow win an award?  
Because he was outstanding in his field! 🌾

User: Tell me another joke, similar to one before.

Assistant: <think>
</think>
好的，我们来讲一个类似风格的笑话。这个笑话同样基于“数字”和“语言”的巧妙结合，希望您会喜欢。
---
### **笑话：关于“数”与“说”**
有一天，一位数学家走进了一家酒吧。他点了一杯啤酒，然后对酒保说：
**“你知道吗？我刚刚在心里算了算，如果我把这杯啤酒喝完，再加上你给我的小费，那么‘我’（I）就变成了‘二’（2）。”**
酒保愣了一下，然后若有所思地点点头。
数学家继续说：“而且更有趣的是，如果你把这句话翻译成中文——‘你知道吗？我刚刚在心里算了算，如果我把这杯啤酒喝完，再加上你给我的小费，那么‘我’（Wǒ）就变成了‘二’（Èr）。”——那么整个句子就变成了一个非常重要的数学定理：**1 + 1 = 2**。
酒保听完后哈哈大笑：“哇！所以你是说，这句话本身就是一个证明？”
数学家微笑着回答：“不仅仅是证明。它还告诉我们，无论用哪种语言表达，逻辑都是通用的。但最重要的是……” 他压低声音说道：“**现在你欠我两杯啤酒了！**”
---
### **为什么这个笑话像之前那些？**
1.  **核心元素相似：** 它依然围绕着“数字”（1, 2）和“语言”（英语/中文）展开。
2.  **双关语：** 利用了“数字”和“人名/代词”发音相似的特点进行幽默。
    *   英语中：`I (1) + 1 = 2 (Two)`
    *   中文中：`我 (Wǒ) + 1 = 二 (Èr)`
3.  **转折与反转：** 从简单的计算转向更深层次的哲学或逻辑思考（1+1=2），并最终将其引申到现实生活中的“债务”，增加了趣味性和意外感。
4.  **文化桥梁：** 它巧妙地连接了东西方文化（英

In [58]:
mgr.add_user('Tell me another joke, different from one before.', metadata = {'think_mode': ' think a bit'})

In [59]:
mgr.build_rwkv7_prompt(user_assistant_count=1)

'User: Tell me another joke, different from one before. think a bit\n\nAssistant: <think'

In [60]:
assistant_output = model.generate(mgr.build_rwkv7_prompt(user_assistant_count=1))
mgr.add_assistant_raw(assistant_output)

Input prompt:  User: Tell me another joke, different from one before. think a bit

Assistant: <think
Output:  >
</think>
好的，这是一个基于《红楼梦》现代职场小说大纲的新笑话：
**新笑话：数据分析师的“红袖添香”**
贾润玉每天加班到深夜，王熙凤（CEO）常在他办公室门口踱步，像等待诗人的佳作。有一天，她终于忍不住问：“润玉啊，你这‘红袖添香’（指熬夜加班）是为了什么？难道你想成为下一个‘金陵十二钗’？”
贾润玉抬起头，眼睛里闪着数据的光芒：“王总，我正在分析用户行为数据。您看，上周五晚上8点到9点，女性用户搜索‘如何让男朋友注意到自己’的关键词量激增了300%。”
王熙凤愣住了：“这……这算什么？难道是因为我们公司没有提供‘红袖添香’服务吗？”
贾润玉微笑着回答：“不，王总。这说明用户需要的不是物理上的陪伴，而是情感上的认可和关注。我们应该推出一个功能：当用户感到孤独时，系统会自动推送一句温暖的话语或一个相关的内容。这才是真正的‘红袖添香’——它不需要人工服务，却能满足人心中最深处的渴望。”
王熙凤若有所思地点点头：“嗯……你说得对。那就按你说的做吧！但记住，别让这项目变成另一个‘镜花水月’！”


In [ ]:
print(mgr.build_prompt())